# Imports

In [62]:
import numpy as np
import matplotlib.pyplot as plt
from ase import Atoms
from ase.build import bulk
from pathlib import Path
from ase.io import read, write
from ase.visualize import view
from ase.build import make_supercell
from ase.build import sort as ase_sort

# Stacking faults (Zinc Blende --> Wurtzite)

# Filling holes (Rock Salt --> Spinel --> Zinc Blende)

In [3]:
# Rock Salt structures
rocksalt_structures = [str(filepath) for filepath in Path('./data/CIFs/').rglob('RockSalt*.cif')]
rocksalt_compositions = [filepath.split('/')[-1].split('_')[-1].split('.')[0] for filepath in rocksalt_structures]
rocksalt_metals = [composition[:-1] for composition in rocksalt_compositions]

# Spinel structures
spinel_structures = [str(filepath) for filepath in Path('./data/CIFs/').rglob('Spinel*.cif')]
spinel_compositions = [filepath.split('/')[-1].split('_')[-1].split('.')[0] for filepath in spinel_structures]
spinel_metals = [composition[: len(composition[:-3]) // 2] for composition in spinel_compositions]

# Zinc Blende structures
zincblende_structures = [str(filepath) for filepath in Path('./data/CIFs/').rglob('ZincBlende*.cif')]
zincblende_compositions = [filepath.split('/')[-1].split('_')[-1].split('.')[0] for filepath in zincblende_structures]
zincblende_metals = [composition[:-1] for composition in zincblende_compositions]


In [57]:
def rocksalt_transformation(rocksalt_unitcell, expansion_factor=2, translation=0.125):
    # Make a supercell of the unit cell
    rocksalt_supercell = make_supercell(rocksalt_unitcell, np.diag([expansion_factor, expansion_factor, expansion_factor]))
    
    # Translate the atoms in the supercell
    rocksalt_supercell.set_scaled_positions(rocksalt_supercell.get_scaled_positions() + translation) 
    
    return rocksalt_supercell

def zincblende_transformation(zincblende_unitcell, expansion_factor=2, inversion=True):
    # Make a supercell of the unit cell
    zincblende_supercell = make_supercell(zincblende_unitcell, np.diag([expansion_factor, expansion_factor, expansion_factor]))
    
    if inversion:
        # Invert the atoms in the supercell
        zincblende_supercell.set_scaled_positions(1 - zincblende_supercell.get_scaled_positions())
    
    return zincblende_supercell

In [58]:
# Read and visualize the structures for the same metal

metal_of_interest = 'Fe'

# Rock Salt
rocksalt_structure = read(rocksalt_structures[rocksalt_metals.index(metal_of_interest)])
rocksalt_transformed = rocksalt_transformation(rocksalt_structure)

# Spinel
spinel_structure = read(spinel_structures[spinel_metals.index(metal_of_interest)])

# Zinc Blende
zincblende_structure = read(zincblende_structures[zincblende_metals.index(metal_of_interest)])
zincblende_transformed = zincblende_transformation(zincblende_structure)

In [59]:
zincblende_transformed.get_scaled_positions()

array([[0.   , 0.   , 0.   ],
       [0.   , 0.75 , 0.75 ],
       [0.75 , 0.   , 0.75 ],
       [0.75 , 0.75 , 0.   ],
       [0.875, 0.875, 0.875],
       [0.625, 0.625, 0.875],
       [0.625, 0.875, 0.625],
       [0.875, 0.625, 0.625],
       [0.   , 0.   , 0.5  ],
       [0.   , 0.75 , 0.25 ],
       [0.75 , 0.   , 0.25 ],
       [0.75 , 0.75 , 0.5  ],
       [0.875, 0.875, 0.375],
       [0.625, 0.625, 0.375],
       [0.625, 0.875, 0.125],
       [0.875, 0.625, 0.125],
       [0.   , 0.5  , 0.   ],
       [0.   , 0.25 , 0.75 ],
       [0.75 , 0.5  , 0.75 ],
       [0.75 , 0.25 , 0.   ],
       [0.875, 0.375, 0.875],
       [0.625, 0.125, 0.875],
       [0.625, 0.375, 0.625],
       [0.875, 0.125, 0.625],
       [0.   , 0.5  , 0.5  ],
       [0.   , 0.25 , 0.25 ],
       [0.75 , 0.5  , 0.25 ],
       [0.75 , 0.25 , 0.5  ],
       [0.875, 0.375, 0.375],
       [0.625, 0.125, 0.375],
       [0.625, 0.375, 0.125],
       [0.875, 0.125, 0.125],
       [0.5  , 0.   , 0.   ],
       [0.

In [48]:
write('rocksalt_test.cif', rocksalt_transformed)
write('spinel_test.cif', spinel_structure)
write('zincblende_test.cif', zincblende_transformed)

In [61]:
len(rocksalt_transformed), len(spinel_structure), len(zincblende_transformed)

(64, 56, 64)

In [60]:
view([rocksalt_transformed, spinel_structure, zincblende_transformed], viewer='ngl')

In [68]:
# Define interpolation functions for the structures
def rocksalt_to_spinel(rocksalt_structure, spinel_structure, n_steps=10):
    # Sort the atoms by x, y, z coordinate values
    # rocksalt_structure = ase_sort(rocksalt_structure, tags=rocksalt_structure.get_scaled_positions())
    # spinel_structure = ase_sort(spinel_structure, tags=spinel_structure.get_scaled_positions())    
    
    rocksalt_positions = rocksalt_structure.get_scaled_positions()
    spinel_positions = spinel_structure.get_scaled_positions()
    
    rocksalt_atoms = rocksalt_structure.get_atomic_numbers()
    spinel_atoms = spinel_structure.get_atomic_numbers()
    
    rocksalt_O_positions = rocksalt_positions[rocksalt_atoms == 8]
    spinel_O_positions = spinel_positions[spinel_atoms == 8]
    
    # Find the closest atoms between the two structures
    dist_matrix = np.linalg.norm(rocksalt_positions[:, None] - spinel_positions, axis=-1)
    # Match the closest atoms in the two structures by finding the minimum distance between unique atoms
    
    
    print(dist_matrix)
    
    # interpolated_structures = []
    # for i in range(n_steps):
    #     # Interpolate the positions of closest oxygen atoms
    #     interpolated_O_positions = rocksalt_O_positions + i / n_steps * (spinel_O_positions - rocksalt_O_positions)
        
    #     # Remove some metal atoms from the rocksalt structure
        
        
    
    return None #interpolated_structures

def zincblende_to_spinel(zincblende_structure, spinel_structure, n_steps=10):
    zincblende_positions = zincblende_structure.get_positions()
    spinel_positions = spinel_structure.get_positions()
    
    interpolated_structures = []
    for i in range(n_steps):
        interpolated_positions = zincblende_positions + i / n_steps * (spinel_positions - zincblende_positions)
        interpolated_structure = Atoms(numbers=zincblende_structure.get_atomic_numbers(), positions=interpolated_positions)
        interpolated_structures.append(interpolated_structure)
    
    return interpolated_structures

In [69]:
rocksalt_to_spinel(rocksalt_transformed, spinel_structure)

[[0.64951905 0.4145781  0.4145781  ... 0.23824626 1.06984171 0.84791584]
 [0.4145781  0.64951905 0.4145781  ... 0.41153527 0.88428575 0.57581358]
 [0.4145781  0.4145781  0.4145781  ... 0.23824626 0.88428575 0.57581358]
 ...
 [0.4145781  0.96014322 1.08253175 ... 0.85910493 0.35420514 0.60254567]
 [0.4145781  1.08253175 1.08253175 ... 0.92231301 0.35420514 0.60254567]
 [0.4145781  1.08253175 0.96014322 ... 0.92231301 0.61274895 0.33624586]]


In [17]:
# Visualize the rock salt structure
# view(rocksalt_structure, viewer='ngl', repeat=(2,2,2))

In [16]:
# view(spinel_structure, viewer='ngl')

In [15]:
# view(zincblende_structure, viewer='ngl', repeat=(2,2,2))

In [19]:
# Count the number of O atoms in the rock salt structure
rocksalt_n_O = sum(rocksalt_structure.get_atomic_numbers() == 8)
print(f'Number of O atoms in the rock salt structure: {rocksalt_n_O}')

# Count the number of O atoms in a 2x2x2 supercell of the rock salt structure
rocksalt_supercell = make_supercell(rocksalt_structure, np.diag([2, 2, 2]))
rocksalt_n_O_supercell = sum(rocksalt_supercell.get_atomic_numbers() == 8)
print(f'Number of O atoms in a 2x2x2 supercell of the rock salt structure: {rocksalt_n_O_supercell}')

# Count the number of O atoms in the spinel structure
spinel_n_O = sum(spinel_structure.get_atomic_numbers() == 8)
print(f'Number of O atoms in the spinel structure: {spinel_n_O}')

# Count the number of O atoms in the zinc blende structure
zincblende_n_O = sum(zincblende_structure.get_atomic_numbers() == 8)
print(f'Number of O atoms in the zinc blende structure: {zincblende_n_O}')

# Count the number of O atoms in a 2x2x2 supercell of the zinc blende structure
zincblende_supercell = make_supercell(zincblende_structure, np.diag([2, 2, 2]))
zincblende_n_O_supercell = sum(zincblende_supercell.get_atomic_numbers() == 8)
print(f'Number of O atoms in a 2x2x2 supercell of the zinc blende structure: {zincblende_n_O_supercell}')

Number of O atoms in the rock salt structure: 4
Number of O atoms in a 2x2x2 supercell of the rock salt structure: 32
Number of O atoms in the spinel structure: 32
Number of O atoms in the zinc blende structure: 4
Number of O atoms in a 2x2x2 supercell of the zinc blende structure: 32


In [24]:
# Show the scaled positions of the O atoms in the rock salt structure
rocksalt_O_mask = rocksalt_supercell.get_atomic_numbers() == 8
rocksalt_O_positions = rocksalt_supercell.get_scaled_positions()[rocksalt_O_mask]
rocksalt_O_positions

array([[0.25, 0.25, 0.25],
       [0.25, 0.  , 0.  ],
       [0.  , 0.25, 0.  ],
       [0.  , 0.  , 0.25],
       [0.25, 0.25, 0.75],
       [0.25, 0.  , 0.5 ],
       [0.  , 0.25, 0.5 ],
       [0.  , 0.  , 0.75],
       [0.25, 0.75, 0.25],
       [0.25, 0.5 , 0.  ],
       [0.  , 0.75, 0.  ],
       [0.  , 0.5 , 0.25],
       [0.25, 0.75, 0.75],
       [0.25, 0.5 , 0.5 ],
       [0.  , 0.75, 0.5 ],
       [0.  , 0.5 , 0.75],
       [0.75, 0.25, 0.25],
       [0.75, 0.  , 0.  ],
       [0.5 , 0.25, 0.  ],
       [0.5 , 0.  , 0.25],
       [0.75, 0.25, 0.75],
       [0.75, 0.  , 0.5 ],
       [0.5 , 0.25, 0.5 ],
       [0.5 , 0.  , 0.75],
       [0.75, 0.75, 0.25],
       [0.75, 0.5 , 0.  ],
       [0.5 , 0.75, 0.  ],
       [0.5 , 0.5 , 0.25],
       [0.75, 0.75, 0.75],
       [0.75, 0.5 , 0.5 ],
       [0.5 , 0.75, 0.5 ],
       [0.5 , 0.5 , 0.75]])

In [27]:
# Show the scaled positions of the O atoms in the spinel structure
spinel_O_mask = spinel_structure.get_atomic_numbers() == 8
spinel_O_positions = spinel_structure.get_scaled_positions()[spinel_O_mask]
spinel_O_positions

array([[0.3626, 0.3626, 0.3626],
       [0.6374, 0.1374, 0.8626],
       [0.1374, 0.8626, 0.6374],
       [0.8626, 0.6374, 0.1374],
       [0.1126, 0.6126, 0.3874],
       [0.8874, 0.8874, 0.8874],
       [0.6126, 0.3874, 0.1126],
       [0.3874, 0.1126, 0.6126],
       [0.6126, 0.1126, 0.3874],
       [0.1126, 0.3874, 0.6126],
       [0.3874, 0.6126, 0.1126],
       [0.1374, 0.6374, 0.8626],
       [0.6374, 0.8626, 0.1374],
       [0.8626, 0.1374, 0.6374],
       [0.3626, 0.8626, 0.8626],
       [0.6374, 0.6374, 0.3626],
       [0.1374, 0.3626, 0.1374],
       [0.1126, 0.1126, 0.8874],
       [0.8874, 0.3874, 0.3874],
       [0.6126, 0.8874, 0.6126],
       [0.6126, 0.6126, 0.8874],
       [0.1126, 0.8874, 0.1126],
       [0.1374, 0.1374, 0.3626],
       [0.6374, 0.3626, 0.6374],
       [0.8626, 0.3626, 0.8626],
       [0.3626, 0.6374, 0.6374],
       [0.3874, 0.8874, 0.3874],
       [0.8874, 0.1126, 0.1126],
       [0.8874, 0.6126, 0.6126],
       [0.3626, 0.1374, 0.1374],
       [0.

In [28]:
# Show the scaled positions of the O atoms in the zinc blende structure
zincblende_O_mask = zincblende_supercell.get_atomic_numbers() == 8
zincblende_O_positions = zincblende_supercell.get_scaled_positions()[zincblende_O_mask]
zincblende_O_positions

array([[0.125, 0.125, 0.125],
       [0.375, 0.375, 0.125],
       [0.375, 0.125, 0.375],
       [0.125, 0.375, 0.375],
       [0.125, 0.125, 0.625],
       [0.375, 0.375, 0.625],
       [0.375, 0.125, 0.875],
       [0.125, 0.375, 0.875],
       [0.125, 0.625, 0.125],
       [0.375, 0.875, 0.125],
       [0.375, 0.625, 0.375],
       [0.125, 0.875, 0.375],
       [0.125, 0.625, 0.625],
       [0.375, 0.875, 0.625],
       [0.375, 0.625, 0.875],
       [0.125, 0.875, 0.875],
       [0.625, 0.125, 0.125],
       [0.875, 0.375, 0.125],
       [0.875, 0.125, 0.375],
       [0.625, 0.375, 0.375],
       [0.625, 0.125, 0.625],
       [0.875, 0.375, 0.625],
       [0.875, 0.125, 0.875],
       [0.625, 0.375, 0.875],
       [0.625, 0.625, 0.125],
       [0.875, 0.875, 0.125],
       [0.875, 0.625, 0.375],
       [0.625, 0.875, 0.375],
       [0.625, 0.625, 0.625],
       [0.875, 0.875, 0.625],
       [0.875, 0.625, 0.875],
       [0.625, 0.875, 0.875]])

In [41]:
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go

# Create a DataFrame for the O positions in the rock salt structure
rocksalt_O_df = pd.DataFrame(rocksalt_O_positions + 0.125, columns=['x', 'y', 'z'])
rocksalt_O_df['structure'] = 'Rock Salt'

# Create a DataFrame for the O positions in the spinel structure
spinel_O_df = pd.DataFrame(spinel_O_positions, columns=['x', 'y', 'z'])
spinel_O_df['structure'] = 'Spinel'

# Create a DataFrame for the O positions in the zinc blende structure
zincblende_O_df = pd.DataFrame(1 - zincblende_O_positions, columns=['x', 'y', 'z'])
zincblende_O_df['structure'] = 'Zinc Blende'

# Concatenate the DataFrames
O_df = pd.concat([rocksalt_O_df, spinel_O_df, zincblende_O_df])

In [42]:
# Create a 3d scatter plot
fig = px.scatter_3d(O_df, x='x', y='y', z='z', color='structure', title='Oxygen positions in different structures')
fig.show()

# Layer stacking (Cadmium Chloride --> Cadmium Iodide)